In [ ]:
TRAINING_DATASET = "synth/prev40_dim4_mixed/train.csv"
FEATURE_MAP = "sps/toy_features_4_mixed.json"
SEED = 4
N_SUB = 1000
N_BOOT = 100
N_PERM = 100
SUB_FRAC = 0.8

Q_ALPHA = 0.05

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'
  SRC = f"{PROJECT_ROOT}/src"
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
  if SRC not in sys.path:
    sys.path.append(SRC)

In [ ]:
import json
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif
from sklearn.utils import resample
from notebooks.analysis_utils import CustPalette
from src.config import Config
from src.cevaehe_new.causal_validation import run_test_classifier

rng = np.random.default_rng(seed=SEED)

In [ ]:
training_dataset = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{TRAINING_DATASET}')
with open(f"{PROJECT_ROOT}/configs/{FEATURE_MAP}", 'r') as f:
  feature_map = json.load(f)

In [ ]:
target_col = feature_map['sens'][0]['name']
feature_cols = [col['name'] for col in (feature_map['X'] + feature_map['ind'])]
dtypes = {col['name']:col['type'] for col in (feature_map['X'] + feature_map['ind'])}
discrete_mask = np.array([dtypes[f] in ("binary", "categorical") for f in feature_cols])

X = training_dataset[feature_cols].to_numpy()
s_target = training_dataset[target_col].to_numpy()

print(f"{len(feature_cols)} features | {discrete_mask.sum()} discrete, {(~discrete_mask).sum()} continuous")

In [ ]:
n_pop = len(training_dataset)
n_sub = int(SUB_FRAC * n_pop)
boot_scores = np.empty((N_BOOT, len(feature_cols)))

for i in range(N_BOOT):
  idx = rng.choice(n_pop, size=n_sub, replace=False)
  boot_scores[i] = mutual_info_classif(
    X[idx], s_target[idx], discrete_features=discrete_mask, n_neighbors=3, random_state=SEED + i
  )

mean_score = boot_scores.mean(axis=0)

In [ ]:
s_shuffled = np.tile(s_target, (N_PERM, 1))
rng.permuted(s_shuffled, axis=1, out=s_shuffled)

null_scores = np.stack([
  mutual_info_classif(X, s_shuffled[i], discrete_features=discrete_mask,
                        n_neighbors=3, random_state=SEED)
  for i in range(N_PERM)
])
q_null = np.percentile(null_scores, (1 - Q_ALPHA)*100, axis=0)
p_val = np.mean(null_scores > mean_score, axis=0)

In [ ]:
from statsmodels.stats.multitest import multipletests

mi_df = pd.DataFrame({
  "feature": feature_cols,
  "MI mean": mean_score,
  "MI low q": np.percentile(boot_scores, Q_ALPHA*100, axis=0),
  "MI high q": np.percentile(boot_scores, (1 - Q_ALPHA)*100, axis=0),
  "Null Baseline high q":q_null,
  "p-val": p_val,
}).sort_values(["MI mean"], ascending=[False]).reset_index(drop=True)

reject, pvals_corrected, _, _ = multipletests(mi_df["p-val"], alpha=Q_ALPHA, method='bonferroni')
mi_df["p-val_adj"] = pvals_corrected
mi_df["Xind_adj"] = (mi_df["MI mean"] < mi_df["Null Baseline high q"]) | (mi_df["p-val_adj"] > Q_ALPHA)

print(mi_df.to_markdown(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.35 * len(mi_df) + 1))
sns.barplot(data=mi_df, y="feature", x="MI mean",
            dodge=False, ax=ax)

ax.scatter(mi_df["Null Baseline high q"], np.arange(len(mi_df)), color="black", marker="|", s=200,
           label="Null Baseline high q", zorder=5)
ax.set_xlabel("Estimated MI(X, S) (nats)")
ax.set_ylabel("")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()